In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr, spearmanr
from statsmodels.stats.contingency_tables import mcnemar


def analysis_pipeline(path):
    log_file = f'{path}/log.txt'


    # ==========================================
    # 0. 全局设置与数据准备
    # ==========================================
    # 设置学术级绘图风格：去除多余网格线，增加刻度线
    sns.set_theme(style="ticks", context="paper", font_scale=1.2)

    # --- 替换为你自己的数据读取代码 ---
    df = pd.read_csv(f'{path}/logit_compare.csv')

    # 假设这里是概率值，如果是 Logit，阈值应设为 0
    THRESHOLD = 0.5 

    with open(log_file, 'w') as f:
        f.write(f"数据总样本量: {len(df)}\n\n")


    # ==========================================
    # 1. 散点图与相关性分析 (连续值视角)
    # ==========================================
    plt.figure(figsize=(6, 6))
    sns.scatterplot(
        data=df, x='esm', y='pf', hue='label', 
        palette=['#4C72B0', '#C44E52'],  # 使用柔和的对比色
        alpha=0.6, edgecolor=None, s=30
    )

    # 绘制对角线 y=x
    plt.plot([0, 1], [0, 1], 'k--', lw=1.5, alpha=0.5, label='y=x (Perfect Agreement)')

    plt.title('Prediction Correlation: ESM vs PF')
    plt.xlabel('ESM Prediction')
    plt.ylabel('PF Prediction')
    plt.legend(title='Label', loc='best')
    sns.despine() # 移除上方和右侧边框 (学术风常见)
    plt.tight_layout()
    plt.savefig(f'{path}/1_scatter_correlation.jpg', dpi=300)
    plt.close()

    # 计算相关系数
    pearson_corr, _ = pearsonr(df['esm'], df['pf'])
    spearman_corr, _ = spearmanr(df['esm'], df['pf'])
    with open(log_file, 'a') as f:
        f.write(f"--- 1. 相关性分析 ---\n")
        f.write(f"Pearson 相关系数 (r): {pearson_corr:.4f}\n")
        f.write(f"Spearman 相关系数 (rho): {spearman_corr:.4f}\n\n")


    # ==========================================
    # 2. 模型间一致性矩阵与 McNemar 检验 (离散值视角)
    # ==========================================
    # 二值化预测结果
    df['esm_pred_class'] = (df['esm'] >= THRESHOLD).astype(int)
    df['pf_pred_class'] = (df['pf'] >= THRESHOLD).astype(int)

    # 计算各自的准确与否 (1为对，0为错)
    df['esm_correct'] = (df['esm_pred_class'] == df['label']).astype(int)
    df['pf_correct'] = (df['pf_pred_class'] == df['label']).astype(int)
    # print(sum(df['esm_correct']), sum(df['pf_correct']))

    # 构建 2x2 列联表 (Contingency Table)
    #         PF 对   PF 错
    # ESM 对   N11     N10
    # ESM 错   N01     N00
    crosstab = pd.crosstab(df['esm_correct'], df['pf_correct'], 
                        rownames=['ESM Correct'], colnames=['PF Correct'])
    crosstab.to_csv(f'{path}/2_crosstab.csv')
    # with open(log_file, 'a') as f:
    #     f.write(f"--- 2. 模型间一致性矩阵 ---\n")
    #     f.write(crosstab)
    #     f.write("\n")

    # McNemar's Test 检验差异显著性
    # 提取非对角线元素 (模型分歧的样本)
    assert crosstab.shape == (2, 2)
    n01 = crosstab.iloc[0, 1] # ESM 错, PF 对
    n10 = crosstab.iloc[1, 0] # ESM 对, PF 错
    # print("都对的数量：", crosstab.iloc[1, 1])
    table = [[crosstab.iloc[1, 1], n10],
            [n01, crosstab.iloc[0, 0]]]
    
    result = mcnemar(table, exact=False, correction=True)
    
    with open(log_file, 'a') as f:
        f.write("--- 2. McNemar 检验 (离散值视角) ---\n")
        f.write(f"McNemar's Test 统计量: {result.statistic:.4f}, p-value: {result.pvalue:.4e}\n")
        if result.pvalue < 0.05:
            f.write("结论: 两个模型在预测分歧上存在显著的统计学差异 (非随机波动)。\n\n")
        else:
            f.write("结论: 两个模型的分歧可能是随机造成的，预测倾向无显著差异。\n\n")


    # ==========================================
    # 3. 预测差异分布与残差分析 (Margin Distribution)
    # ==========================================
    df['pred_diff'] = np.abs(df['esm'] - df['pf'])

    plt.figure(figsize=(7, 4))
    sns.histplot(df, x='pred_diff', hue='label', bins=30, kde=True, 
                palette=['#4C72B0', '#C44E52'], element="step")
    plt.title('Distribution of Absolute Prediction Differences (|ESM - PF|)')
    plt.xlabel('Absolute Difference')
    plt.ylabel('Count')
    sns.despine()
    plt.tight_layout()
    plt.savefig(f'{path}/3_difference_distribution.jpg', dpi=300)
    plt.close()


    # ==========================================
    # 4. 提取分歧子集 (Error Analysis)
    # ==========================================
    # 提取 ESM 对但 PF 错的样本
    subset_esm_better = df[(df['esm_correct'] == 1) & (df['pf_correct'] == 0)].copy()
    # 提取 PF 对但 ESM 错的样本
    subset_pf_better = df[(df['pf_correct'] == 1) & (df['esm_correct'] == 0)].copy()

    with open(log_file, 'a') as f:
        f.write(f"--- 4. 分歧子集提取 ---\n")
        f.write(f"ESM 模型具有优势的样本数: {len(subset_esm_better)}\n")
        f.write(f"PF 模型具有优势的样本数: {len(subset_pf_better)}\n\n")

    # 提取极端困难样本：Label为1，但两个模型都预测得非常低 (< 0.2)
    hard_positives = df[(df['label'] == 1) & (df['esm'] < 0.2) & (df['pf'] < 0.2)]
    with open(log_file, 'a') as f:
        f.write(f"两个模型共同失效的 Hard Positives 样本数: {len(hard_positives)}\n")

    # 将这些特定的子集保存出来，便于后续做生物学或特征层面的 case study
    # subset_esm_better.to_csv('subset_esm_better.csv', index=False)
    # subset_pf_better.to_csv('subset_pf_better.csv', index=False)
    # hard_positives.to_csv('hard_positives.csv', index=False)

In [9]:
import pandas as pd
import itertools
import os

cell_lines = ['A549', 'HSC5', 'MELJUSO']
scenes = ['C1', 'C2', 'Tail']
folds = list(range(5))

for cl, scene in itertools.product(cell_lines, scenes):
    out_folder = f'./comparison/{cl}_{scene}'
    os.makedirs(out_folder, exist_ok=True)

    esm = pd.read_csv(f'/data/qingyuyang/SL/ESM4SL/output/attn_len1000_layer1_meanpool/{cl}/{scene}/fold_0/csv_log/version_0/test_logits.csv')
    esm.rename({'probs': 'esm'}, axis=1, inplace=True)
    pf = pd.read_csv(f'/data/qingyuyang/SL/PFSL/output/pf_biolinkbert/{cl}/{scene}/fold_0_logits.csv')
    pf.rename({'probs': 'pf'}, axis=1, inplace=True)
    gt = pd.read_csv(f'/data/qingyuyang/SL/PFSL/data/SLKB/specific/{cl}/{scene}/sl_test_0.csv')
    gt.rename({'2': 'label'}, axis=1, inplace=True)

    res_comb = pd.merge(esm, pf, on=['gene1', 'gene2'])
    res_comb = pd.merge(res_comb, gt, left_on=['gene1', 'gene2'], right_on=['0', '1'])
    res_comb.drop(labels=['0', '1'], axis=1, inplace=True)
    # print(res_comb)
    res_comb.to_csv(f'{out_folder}/logit_compare.csv', index=False)

    analysis_pipeline(out_folder)
    # break